# ECG Digitizer - Offline Baseline Submission

This notebook runs the ECG digitizer in offline mode using pre-packaged dependencies.

## Requirements
1. Upload ECG Digitizer source code and weights as a Kaggle Dataset
2. Attach the dataset to this notebook
3. Run all cells to generate `submission.csv`

## Dataset Structure Expected:
```
/kaggle/input/ecg-digitizer/
├── src/
│   ├── model/
│   ├── config/
│   └── kaggle_inference.py
└── weights/
    ├── unet_best.pth
    └── leadnet_best.pth
```

## Cell 1: Setup Environment

In [ ]:
import sys
import os
from pathlib import Path
import shutil

# Setup paths
DATASET_PATH = Path('/kaggle/input/ecg-digitizer-source-code-and-weights')
WORKING_DIR = Path('/kaggle/working')

print("Setting up environment...")

# Copy source code to working directory
if (DATASET_PATH / 'src').exists():
    shutil.copytree(DATASET_PATH / 'src', WORKING_DIR / 'src', dirs_exist_ok=True)
    print(f"✅ Copied source code from dataset")
else:
    print("❌ Dataset not found! Make sure you've attached the ecg-digitizer dataset")
    print("   Go to 'Add Data' → Search for your uploaded dataset → Add")
    sys.exit(1)

# Copy weights to working directory
if (DATASET_PATH / 'weights').exists():
    shutil.copytree(DATASET_PATH / 'weights', WORKING_DIR / 'weights', dirs_exist_ok=True)
    print(f"✅ Copied model weights from dataset")
else:
    print("❌ Weights not found in dataset!")
    sys.exit(1)

# Add to Python path
sys.path.insert(0, str(WORKING_DIR))

print("\n✅ Environment setup complete!")

## Cell 2: Verify Dependencies

In [ ]:
# Verify required packages (should be available in Kaggle by default)
import numpy as np
import scipy
import torch
import cv2
import pandas as pd
import matplotlib
from sklearn import __version__ as sklearn_version

print("Package versions:")
print(f"  NumPy: {np.__version__}")
print(f"  SciPy: {scipy.__version__}")
print(f"  PyTorch: {torch.__version__}")
print(f"  OpenCV: {cv2.__version__}")
print(f"  Pandas: {pd.__version__}")
print(f"  Matplotlib: {matplotlib.__version__}")
print(f"  Scikit-learn: {sklearn_version}")

# Check CUDA availability
if torch.cuda.is_available():
    print(f"\n✅ CUDA available: {torch.cuda.get_device_name(0)}")
else:
    print("\n⚠️ CUDA not available, using CPU (slower but will work)")

# ===================================================================
# YACS COMPATIBILITY SHIM - Minimal implementation for offline mode
# ===================================================================
# The source code uses yacs for configuration, but it's not available
# in Kaggle offline mode. This provides a minimal compatible implementation.

class CfgNode(dict):
    """Minimal yacs.config.CfgNode replacement"""
    
    def __init__(self, init_dict=None, key_list=None, new_allowed=True):
        init_dict = {} if init_dict is None else init_dict
        super().__init__(init_dict)
        self.__dict__['__immutable__'] = False
        self.__dict__['__new_allowed__'] = new_allowed
    
    def __getattr__(self, name):
        if name in self:
            return self[name]
        else:
            raise AttributeError(f"CfgNode has no attribute '{name}'")
    
    def __setattr__(self, name, value):
        if self.__dict__.get('__immutable__', False):
            raise AttributeError("Cannot modify immutable config")
        self[name] = value
    
    def clone(self):
        return CfgNode(dict(self))
    
    def freeze(self):
        self.__dict__['__immutable__'] = True
        for v in self.values():
            if isinstance(v, CfgNode):
                v.freeze()
    
    def defrost(self):
        self.__dict__['__immutable__'] = False
        for v in self.values():
            if isinstance(v, CfgNode):
                v.defrost()
    
    def set_new_allowed(self, is_new_allowed):
        """Allow or disallow adding new keys."""
        self.__dict__['__new_allowed__'] = is_new_allowed
    
    def merge_from_list(self, cfg_list):
        """Merge config from list [key, value, key, value, ...]"""
        if len(cfg_list) % 2 != 0:
            raise ValueError(f"Override list has odd length: {len(cfg_list)}")
        
        for i in range(0, len(cfg_list), 2):
            key_path = cfg_list[i]
            value = cfg_list[i + 1]
            
            # Convert value string to appropriate type
            try:
                import ast
                value = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                pass
            
            # Navigate nested structure using dot notation
            keys = key_path.split('.')
            current = self
            for key in keys[:-1]:
                if key not in current:
                    current[key] = CfgNode()
                current = current[key]
            
            current[keys[-1]] = value
    
    def _merge_dict(self, cfg_dict):
        """Recursively merge a dictionary into this config."""
        for k, v in cfg_dict.items():
            if isinstance(v, dict):
                if k not in self or not isinstance(self[k], CfgNode):
                    self[k] = CfgNode()
                self[k]._merge_dict(v)
            else:
                self[k] = v
    
    def merge_from_file(self, cfg_filename):
        import yaml
        with open(cfg_filename, 'r') as f:
            cfg_dict = yaml.safe_load(f)
        self._merge_dict(cfg_dict)
    
    def merge_from_other_cfg(self, cfg_other):
        self.update(cfg_other)

# Inject into sys.modules so imports work
import sys
import types

yacs_module = types.ModuleType('yacs')
yacs_config_module = types.ModuleType('yacs.config')
yacs_config_module.CfgNode = CfgNode
yacs_module.config = yacs_config_module
sys.modules['yacs'] = yacs_module
sys.modules['yacs.config'] = yacs_config_module

print("\n✅ All dependencies verified!")
print("✅ YACS compatibility shim installed!")

## Cell 3: Configure Inference (Baseline Mode)

In [ ]:
from yacs.config import CfgNode as CN
from pathlib import Path

print("Loading base configuration from kaggle_inference.yml...")

# Load complete config from YAML
config_path = WORKING_DIR / 'src' / 'config' / 'kaggle_inference.yml'
config = CN()
config.merge_from_file(str(config_path))
print(f"✅ Loaded config from {config_path}")

# Override paths for offline environment
print("\nOverriding paths for offline mode...")

# Update weight paths to use WORKING_DIR
config.MODEL.KWARGS.config.SEGMENTATION_MODEL.weight_path = str(
    WORKING_DIR / 'weights' / 'unet_weights_07072025.pt'
)
config.MODEL.KWARGS.config.LAYOUT_IDENTIFIER.unet_weight_path = str(
    WORKING_DIR / 'weights' / 'lead_name_unet_weights_07072025.pt'
)

# Update config file paths to use WORKING_DIR
config.MODEL.KWARGS.config.LAYOUT_IDENTIFIER.config_path = str(
    WORKING_DIR / 'src' / 'config' / 'lead_layouts_george-moody-2024.yml'
)
config.MODEL.KWARGS.config.LAYOUT_IDENTIFIER.unet_config_path = str(
    WORKING_DIR / 'src' / 'config' / 'lead_name_unet.yml'
)

# Set device based on CUDA availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
config.MODEL.KWARGS.device = device
config.MODEL.KWARGS.config.LAYOUT_IDENTIFIER.KWARGS.device = device

# Update data paths for competition (USER MUST UPDATE THESE!)
config.DATA.test_csv_path = '/kaggle/input/physionet-ecg-image-digitization/test.csv'
config.DATA.test_images_dir = '/kaggle/input/physionet-ecg-image-digitization/test/'
config.DATA.submission_path = str(WORKING_DIR / 'submission.csv')

# BASELINE MODE: Disable TTA and constraints for reliability
config.STRATEGIES.use_tta = False
config.STRATEGIES.tta_n_augmentations = 0
config.STRATEGIES.use_physiological_constraints = False
config.STRATEGIES.constraint_alpha = 0.0

# Validate critical config sections
assert hasattr(config, 'MODEL'), "MODEL section missing!"
assert hasattr(config.MODEL, 'class_path'), "MODEL.class_path missing!"
assert hasattr(config.MODEL, 'KWARGS'), "MODEL.KWARGS missing!"

print("\nConfiguration Summary:")
print(f"  Device: {config.MODEL.KWARGS.device}")
print(f"  UNet weights: {Path(config.MODEL.KWARGS.config.SEGMENTATION_MODEL.weight_path).name}")
print(f"  LeadNet weights: {Path(config.MODEL.KWARGS.config.LAYOUT_IDENTIFIER.unet_weight_path).name}")
print(f"  TTA: {config.STRATEGIES.use_tta}")
print(f"  Constraints: {config.STRATEGIES.use_physiological_constraints}")
print(f"  Test CSV: {config.DATA.test_csv_path}")
print(f"  Output: {config.DATA.submission_path}")
print("\n✅ Baseline configuration ready!")

## Cell 4: Load Models and Run Inference

**IMPORTANT:** Update the test data paths in Cell 3 before running this cell!
- `config.DATA.test_csv` should point to the competition's `test.csv`
- `config.DATA.test_images_dir` should point to the test images directory

In [ ]:
from src.kaggle_inference import main

print("Starting ECG digitizer inference...")
print("="*60)

try:
    # Run inference
    main(config)
    
    print("="*60)
    print("✅ Inference completed successfully!")
    
    # Verify output
    if Path(config.DATA.submission_path).exists():
        import pandas as pd
        df = pd.read_csv(config.DATA.submission_path)
        print(f"\nSubmission file created: {config.DATA.submission_path}")
        print(f"  Rows: {len(df):,}")
        print(f"  Columns: {list(df.columns)}")
        print(f"  Value range: [{df['value'].min():.6f}, {df['value'].max():.6f}]")
        print(f"  NaN count: {df['value'].isna().sum()}")
        
        if df['value'].isna().any():
            print("\n⚠️ WARNING: Submission contains NaN values!")
        else:
            print("\n✅ Submission looks good! Ready to submit.")
    else:
        print(f"\n❌ ERROR: Submission file not created at {config.DATA.submission_path}")
        
except Exception as e:
    print(f"\n❌ ERROR during inference: {e}")
    import traceback
    traceback.print_exc()
    raise

## Cell 5: Validate Submission Format

In [ ]:
import pandas as pd

# Load and validate submission
submission = pd.read_csv(config.DATA.submission_path)

print("Submission validation:")
print(f"  Shape: {submission.shape}")
print(f"  Columns: {list(submission.columns)}")

# Check required columns
required_cols = ['id', 'value']
if all(col in submission.columns for col in required_cols):
    print(f"  ✅ Required columns present: {required_cols}")
else:
    print(f"  ❌ Missing required columns!")

# Check for NaN values
nan_count = submission['value'].isna().sum()
if nan_count == 0:
    print(f"  ✅ No NaN values")
else:
    print(f"  ❌ {nan_count} NaN values found ({nan_count/len(submission)*100:.2f}%)")

# Check ID format (should be: image_id_row_lead)
sample_ids = submission['id'].head(5)
print(f"\nSample IDs:")
for id_val in sample_ids:
    print(f"  {id_val}")

# Statistical summary
print(f"\nValue statistics:")
print(submission['value'].describe())

print("\n" + "="*60)
if nan_count == 0:
    print("✅ Submission is valid and ready to submit!")
    print(f"   File: {config.DATA.submission_path}")
else:
    print("⚠️ Please fix NaN values before submitting")

## Next Steps

1. **Review the validation output above** - Ensure no NaN values and reasonable value ranges
2. **Download submission.csv** from `/kaggle/working/submission.csv`
3. **Submit to competition** via the Kaggle submission interface

---

## Troubleshooting

### Dataset not found
- Make sure you've uploaded the ECG Digitizer files as a Kaggle Dataset
- Attach the dataset to this notebook: Add Data → Your Datasets → ecg-digitizer

### Wrong test data paths
- Update `config.DATA.test_csv` and `config.DATA.test_images_dir` in Cell 3
- These should point to the actual competition test data

### CUDA out of memory
- This shouldn't happen with batch_size=1, but if it does:
- Change `config.INFERENCE.device = 'cpu'` in Cell 3

### NaN values in output
- This usually means lead detection failed for some images
- Check the log output for specific errors
- Consider adjusting detection thresholds in the source code